# Black-box watermark stealing

Goal: learn which token pairs are green without access to the watermark key or detector.

Attacker knowledge:
- 300 watermarked texts (train split) and the matching unwatermarked texts from the same
  open-weight model (LLaDA can be run locally without the watermark),
- the public LLaDA tokenizer,
- the fact that the scheme is a red/green watermark.

Unknown to the attacker: the key (seed), the detector, token colors, and the scheme
parameters (context, gamma, delta).

Rule: the attack sections never import the detector and never read `token_color`,
`mask`, `p_value` or `z_score`. The true key is only used in the final evaluation section.

Runs locally on Mac (CPU).

## 1. Setup and train/test split

In [2]:
import json
import random
from pathlib import Path

import numpy as np
from transformers import AutoTokenizer


def find_project_root(start: Path) -> Path:
    for folder in [start, *start.parents]:
        if (folder / "diffusion-lm-watermark").exists():
            return folder
    raise FileNotFoundError("Project root not found.")


def load_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f]


ROOT = find_project_root(Path.cwd().resolve())
SEED = 42

# The attacker only keeps the prompt, the text and the dataset name
wm = [{"prompt": r["prompt"], "text": r["completion"],
       "dataset": r["dataset_name"].split("_batch_")[0]}
      for r in load_jsonl(ROOT / "results" / "Cleaned Data" / "watermarked_600.jsonl")]
nowm = [{"prompt": r["prompt"], "text": r["completion"],
         "dataset": r["dataset_name"].split("_batch_")[0]}
        for r in load_jsonl(ROOT / "results" / "Cleaned Data" / "unwatermarked_600.jsonl")]

# Both files must list the same prompts in the same order
assert all(a["prompt"] == b["prompt"] for a, b in zip(wm, nowm)), "Prompts are not aligned"

# Split by prompt: 100 train / 100 test per dataset, so a prompt never appears in both
rng = random.Random(SEED)
train_idx, test_idx = [], []
for ds in sorted({r["dataset"] for r in wm}):
    idx = [i for i, r in enumerate(wm) if r["dataset"] == ds]
    rng.shuffle(idx)
    train_idx += idx[:100]
    test_idx += idx[100:]

# Public tokenizer only (no detector)
tokenizer = AutoTokenizer.from_pretrained("GSAI-ML/LLaDA-8B-Instruct", trust_remote_code=True)

def tokenize(text):
    return tokenizer(text, add_special_tokens=False)["input_ids"]

wm_tokens = [tokenize(r["text"]) for r in wm]
nowm_tokens = [tokenize(r["text"]) for r in nowm]

n_train_wm = sum(len(wm_tokens[i]) for i in train_idx)
n_train_nowm = sum(len(nowm_tokens[i]) for i in train_idx)
print(f"Train prompts: {len(train_idx)} | Test prompts: {len(test_idx)}")
print(f"Train tokens: watermarked {n_train_wm:,} | unwatermarked {n_train_nowm:,}")
print(f"Vocabulary size: {len(tokenizer):,}")


/Users/paulmicheli/Desktop/Unine/MA1/Trustworthy/Trustworhty_AI_Watermarking/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Train prompts: 300 | Test prompts: 300
Train tokens: watermarked 73,690 | unwatermarked 67,928
Vocabulary size: 126,349


## 2. Discovery: what does the color depend on?

For each hypothesis, the attacker scores every feature on the train split with a smoothed
log frequency ratio (watermarked vs unwatermarked, normalized by corpus size), then scores
each test text by the mean feature score. A hypothesis is supported if this score separates
watermarked from unwatermarked test texts (ROC AUC well above 0.5).
`unigram` is a control: it captures style differences unrelated to token colors.
Features seen fewer than MIN_COUNT times in train are ignored.

In [3]:
from collections import Counter
from sklearn.metrics import roc_auc_score

MIN_COUNT = 5

HYPOTHESES = {
    "unigram":       lambda t: [(t[i],) for i in range(len(t))],
    "adjacent_pair": lambda t: [(t[i - 1], t[i]) for i in range(1, len(t))],
    "skip_pair":     lambda t: [(t[i - 2], t[i]) for i in range(2, len(t))],
    "triple":        lambda t: [(t[i - 2], t[i - 1], t[i]) for i in range(2, len(t))],
}


def feature_scores(extract, idx):
    """Smoothed log ratio of normalized frequencies, train split only."""
    c_wm = Counter(f for i in idx for f in extract(wm_tokens[i]))
    c_no = Counter(f for i in idx for f in extract(nowm_tokens[i]))
    n_wm, n_no = sum(c_wm.values()), sum(c_no.values())
    scores = {}
    for f in set(c_wm) | set(c_no):
        if c_wm[f] + c_no[f] >= MIN_COUNT:
            scores[f] = np.log((c_wm[f] + 1) / n_wm) - np.log((c_no[f] + 1) / n_no)
    return scores


def text_score(extract, tokens, scores):
    vals = [scores[f] for f in extract(tokens) if f in scores]
    return np.mean(vals) if vals else 0.0, len(vals)


print(f"{'hypothesis':15s} {'features kept':>13s} {'test AUC':>9s} {'test coverage':>14s}")
for name, extract in HYPOTHESES.items():
    scores = feature_scores(extract, train_idx)
    y, s, covered, total = [], [], 0, 0
    for i in test_idx:
        for label, toks in [(1, wm_tokens[i]), (0, nowm_tokens[i])]:
            score, n_known = text_score(extract, toks, scores)
            y.append(label)
            s.append(score)
            covered += n_known
            total += len(extract(toks))
    print(f"{name:15s} {len(scores):13,d} {roc_auc_score(y, s):9.3f} {covered / total:14.1%}")

hypothesis      features kept  test AUC  test coverage
unigram                 3,192     0.725          85.4%
adjacent_pair           4,595     0.862          41.5%
skip_pair               3,464     0.718          32.0%
triple                  2,338     0.746          14.3%


## 3. Discovery, controlled for vocabulary

The unigram control reaches AUC 0.73, so part of every score may come from vocabulary
differences rather than token colors. We subtract the lower-order score:
- pairs: score(a, b) - score(b),
- triples: score(a, b, c) - score(b, c).
If colors depend only on the adjacent token, the controlled adjacent-pair score should
stay well above 0.5, while the controlled skip-pair and triple scores should drop to ~0.5.

In [4]:
uni = feature_scores(HYPOTHESES["unigram"], train_idx)
pair = feature_scores(HYPOTHESES["adjacent_pair"], train_idx)
skip = feature_scores(HYPOTHESES["skip_pair"], train_idx)
tri = feature_scores(HYPOTHESES["triple"], train_idx)

CONTROLLED = {
    "adjacent_pair - unigram": (HYPOTHESES["adjacent_pair"],
                                {f: v - uni.get((f[-1],), 0.0) for f, v in pair.items()}),
    "skip_pair - unigram":     (HYPOTHESES["skip_pair"],
                                {f: v - uni.get((f[-1],), 0.0) for f, v in skip.items()}),
    "triple - adjacent_pair":  (HYPOTHESES["triple"],
                                {f: v - pair.get(f[1:], 0.0) for f, v in tri.items()}),
}

print(f"{'controlled hypothesis':25s} {'test AUC':>9s}")
for name, (extract, scores) in CONTROLLED.items():
    y, s = [], []
    for i in test_idx:
        for label, toks in [(1, wm_tokens[i]), (0, nowm_tokens[i])]:
            y.append(label)
            s.append(text_score(extract, toks, scores)[0])
    print(f"{name:25s} {roc_auc_score(y, s):9.3f}")

controlled hypothesis      test AUC
adjacent_pair - unigram       0.856
skip_pair - unigram           0.674
triple - adjacent_pair        0.611


## 4. Evaluation with the true key (not available to the attacker)

From here on we load the official detector, only to grade the stolen rules.
The true color of a pair (a, b) is obtained by running the detector on the two-token
sequence [a, b]: with C = {-1}, the color of b depends only on a.
Sanity check first: pair-by-pair colors must match the detector's colors on a full text.
Then: among the pairs the attacker believes most green (highest controlled score),
what share is truly green? Random guessing gives gamma = 25%.

In [5]:
import sys
import torch

sys.path.insert(0, str(ROOT / "src"))
from detection import load_detector

_, detector = load_detector(ROOT)


def true_color(a: int, b: int) -> int:
    """True color of token b after token a (1 = green, 0 = red)."""
    with torch.no_grad():
        out = detector.detect(torch.tensor([[a, b]]))
    return int(out["token_color"][0])


# Sanity check on one watermarked text
toks = wm_tokens[test_idx[0]]
with torch.no_grad():
    full = detector.detect(torch.tensor([toks]))["token_color"]
pairwise = [true_color(toks[i - 1], toks[i]) for i in range(1, len(toks))]
print("Sanity check: colors match on a full text:",
      [int(c) for c in full] == pairwise)

Sanity check: colors match on a full text: True


## 5. Precision of the stolen rules

For each of the 4,595 pairs the attacker kept, we get its true color. Pairs are ranked
by the attacker's controlled score (pair score minus unigram score).
- Top-K precision: share of truly green pairs among the K pairs believed most green
  (random baseline 25%).
- Bottom-K: share of truly red pairs among the K pairs believed most red
  (random baseline 75%).
We also check whether precision improves when the attacker saw a pair more often.

In [6]:
pair_ctrl = CONTROLLED["adjacent_pair - unigram"][1]

# Train counts of each kept pair (attacker side)
extract = HYPOTHESES["adjacent_pair"]
c_wm = Counter(f for i in train_idx for f in extract(wm_tokens[i]))
c_no = Counter(f for i in train_idx for f in extract(nowm_tokens[i]))

# True colors (evaluation side, uses the key)
truth = {f: true_color(*f) for f in pair_ctrl}
print(f"Pairs graded: {len(truth):,} | true green share among them: "
      f"{np.mean(list(truth.values())):.1%}")

ranked = sorted(pair_ctrl, key=pair_ctrl.get, reverse=True)

print("\nBelieved green (top-K)        Believed red (bottom-K)")
for k in [50, 100, 250, 500, 1000, 2000]:
    top = np.mean([truth[f] for f in ranked[:k]])
    bottom = 1 - np.mean([truth[f] for f in ranked[-k:]])
    print(f"K={k:5d}: {top:6.1%} green         {bottom:6.1%} red")

print("\nTop-25% of pairs by score, split by how often the attacker saw them")
cutoff = np.quantile(list(pair_ctrl.values()), 0.75)
for lo, hi in [(5, 9), (10, 19), (20, 49), (50, 10**9)]:
    sel = [f for f in pair_ctrl
           if lo <= c_wm[f] + c_no[f] <= hi and pair_ctrl[f] >= cutoff]
    if sel:
        prec = np.mean([truth[f] for f in sel])
        print(f"  seen {lo:>2d}-{hi if hi < 10**9 else '+':>3}: "
              f"{len(sel):5d} pairs, {prec:6.1%} truly green")

Pairs graded: 4,595 | true green share among them: 37.6%

Believed green (top-K)        Believed red (bottom-K)
K=   50:  74.0% green          92.0% red
K=  100:  70.0% green          87.0% red
K=  250:  75.2% green          88.4% red
K=  500:  73.2% green          89.6% red
K= 1000:  67.6% green          87.9% red
K= 2000:  57.5% green          80.8% red

Top-25% of pairs by score, split by how often the attacker saw them
  seen  5-  9:   780 pairs,  61.7% truly green
  seen 10- 19:   258 pairs,  75.2% truly green
  seen 20- 49:    89 pairs,  78.7% truly green
  seen 50-  +:    22 pairs,  95.5% truly green
